# FSL 15-Sign Trainer — Transformer (Video Sign Language / VSL)

Same 15 curated labels, **Transformer encoder** on 30-frame landmark sequences.

**Before running:** add Colab secrets `KAGGLE_USERNAME` and `KAGGLE_KEY`.

**Checkpoints (auto-saved & resumable):**
1. `01_raw_landmarks.npz` — after landmark extraction
2. `02_preprocessed_augmented.npz` + `scaler.pkl` — after augmentations & train/test split
3. `training/best_model.keras` — best weights during training (+ `training_history.json`)

Set `USE_DRIVE = True` to persist checkpoints on Google Drive across sessions.

In [ ]:
!pip -q install mediapipe opencv-python-headless scikit-learn joblib tensorflow pandas

In [ ]:
import os, urllib.request, zipfile
from pathlib import Path

from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

!pip -q install kaggle

KAGGLE_DATASET = 'previlace/fsl-105'
ZIP_PATH = Path('/content/fsl-105.zip')
DATA_ROOT = Path('/content/fsl_dataset')

CHECKPOINT_DIR = Path('/content/checkpoints/fsl_15_transformer')
RAW_CHECKPOINT = CHECKPOINT_DIR / '01_raw_landmarks.npz'
PREPROCESSED_CHECKPOINT = CHECKPOINT_DIR / '02_preprocessed_augmented.npz'
TRAINING_DIR = CHECKPOINT_DIR / 'training'
for p in (CHECKPOINT_DIR, TRAINING_DIR):
    p.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = Path('/content/drive/MyDrive/fsl_15_checkpoints/transformer')
    RAW_CHECKPOINT = CHECKPOINT_DIR / '01_raw_landmarks.npz'
    PREPROCESSED_CHECKPOINT = CHECKPOINT_DIR / '02_preprocessed_augmented.npz'
    TRAINING_DIR = CHECKPOINT_DIR / 'training'
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    TRAINING_DIR.mkdir(parents=True, exist_ok=True)


def resolve_dataset_root(search_root: Path) -> Path:
    if (search_root / 'labels.csv').exists():
        return search_root
    matches = list(search_root.rglob('labels.csv'))
    return matches[0].parent if matches else search_root


def extract_nested_clips_zip(dataset_root: Path):
    videos = list(dataset_root.rglob('*.mov')) + list(dataset_root.rglob('*.MOV'))
    videos += list(dataset_root.rglob('*.mp4')) + list(dataset_root.rglob('*.MP4'))
    if len(videos) > 100:
        return
    for zip_name in ('fsl_clips.zip', 'clips.zip'):
        clips_zip = dataset_root / zip_name
        if clips_zip.exists():
            out_dir = dataset_root / zip_name.replace('.zip', '')
            out_dir.mkdir(parents=True, exist_ok=True)
            print(f'Extracting {zip_name}...')
            with zipfile.ZipFile(clips_zip, 'r') as zf:
                zf.extractall(out_dir)


def download_fsl105() -> Path:
    global DATA_ROOT
    existing = resolve_dataset_root(Path('/content/fsl_dataset'))
    if (existing / 'labels.csv').exists():
        videos = list(existing.rglob('*.mov')) + list(existing.rglob('*.MOV'))
        if len(videos) > 100:
            DATA_ROOT = existing
            print(f'Dataset ready at {DATA_ROOT} ({len(videos)} videos)')
            return DATA_ROOT

    print(f'Downloading {KAGGLE_DATASET}...')
    os.system(f'kaggle datasets download -d {KAGGLE_DATASET} -p /content')
    extract_to = Path('/content/fsl_dataset')
    extract_to.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(extract_to)

    DATA_ROOT = resolve_dataset_root(extract_to)
    extract_nested_clips_zip(DATA_ROOT)
    videos = list(DATA_ROOT.rglob('*.mov')) + list(DATA_ROOT.rglob('*.MOV'))
    print('Dataset root:', DATA_ROOT)
    print('Video clips found:', len(videos))
    return DATA_ROOT


DATA_ROOT = download_fsl105()
print('Checkpoint dir:', CHECKPOINT_DIR)

In [ ]:
import numpy as np

ACTIONS = np.array([
    'GOOD_AFTERNOON', 'NICE_TO_MEET_YOU', 'YES', 'NO', 'THANK_YOU',
    'HOW_ARE_YOU', 'ONE', 'TWO', 'THREE', 'SIX', 'SEVEN',
    'CORRECT', 'WRONG', 'IM_FINE', 'UNDERSTAND',
])
SEQUENCE_LENGTH = 30
KEYPOINT_DIM = 258
NUM_CLASSES = len(ACTIONS)

LABEL_ALIASES = {
    'good afternoon': 'GOOD_AFTERNOON', 'good_afternoon': 'GOOD_AFTERNOON',
    'nice to meet you': 'NICE_TO_MEET_YOU', 'nice_to_meet_you': 'NICE_TO_MEET_YOU',
    'thank you': 'THANK_YOU', 'thank_you': 'THANK_YOU',
    'how are you': 'HOW_ARE_YOU', 'how_are_you': 'HOW_ARE_YOU',
    "i'm fine": 'IM_FINE', 'im fine': 'IM_FINE', 'im_fine': 'IM_FINE',
}

def normalize_label(name: str):
    key = name.strip().replace("'", '').replace('-', ' ')
    upper = '_'.join(key.upper().split())
    if upper in ACTIONS:
        return upper
    lower = key.lower()
    return LABEL_ALIASES.get(lower) or LABEL_ALIASES.get(lower.replace(' ', '_'))

In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks.python.core import base_options as base_options_module
from mediapipe.tasks.python.vision import HolisticLandmarker, HolisticLandmarkerOptions, RunningMode

MODEL_PATH = Path('/content/holistic_landmarker.task')
if not MODEL_PATH.exists():
    url = 'https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/float16/latest/holistic_landmarker.task'
    urllib.request.urlretrieve(url, MODEL_PATH)

landmarker = HolisticLandmarker.create_from_options(HolisticLandmarkerOptions(
    base_options=base_options_module.BaseOptions(model_asset_path=str(MODEL_PATH)),
    running_mode=RunningMode.IMAGE,
    min_pose_detection_confidence=0.5,
    min_pose_landmarks_confidence=0.5,
    min_hand_landmarks_confidence=0.5,
))

def extract_keypoints(result):
    pose = np.zeros(132, dtype=np.float32)
    if result.pose_landmarks:
        for i, lm in enumerate(result.pose_landmarks):
            pose[i*4:(i+1)*4] = [lm.x, lm.y, lm.z, getattr(lm, 'visibility', 0.0)]
    lh = np.zeros(63, dtype=np.float32)
    if result.left_hand_landmarks:
        for i, lm in enumerate(result.left_hand_landmarks):
            lh[i*3:(i+1)*3] = [lm.x, lm.y, lm.z]
    rh = np.zeros(63, dtype=np.float32)
    if result.right_hand_landmarks:
        for i, lm in enumerate(result.right_hand_landmarks):
            rh[i*3:(i+1)*3] = [lm.x, lm.y, lm.z]
    return np.concatenate([pose, lh, rh])

def video_to_sequence(video_path, target_len=30):
    cap = cv2.VideoCapture(str(video_path))
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = landmarker.detect(mp_image)
        frames.append(extract_keypoints(result))
    cap.release()
    if not frames:
        return None
    arr = np.array(frames, dtype=np.float32)
    if len(arr) != target_len:
        idx = np.linspace(0, max(len(arr)-1, 0), target_len).astype(int)
        arr = arr[idx] if len(arr) else None
    return arr

In [ ]:
import pandas as pd


def find_clips_root(fsl_root: Path):
    for root in (fsl_root / 'fsl_clips', fsl_root / 'clips', fsl_root / 'fsl_clips' / 'fsl_clips'):
        if root.exists() and len([p for p in root.iterdir() if p.is_dir() and p.name.isdigit()]) >= 5:
            return root
    best, best_count = None, 0
    for parent in fsl_root.rglob('*'):
        if not parent.is_dir():
            continue
        numeric = [p for p in parent.iterdir() if p.is_dir() and p.name.isdigit()]
        if len(numeric) > best_count:
            best, best_count = parent, len(numeric)
    return best if best_count >= 5 else None


def load_id_to_action(fsl_root: Path):
    labels_csv = fsl_root / 'labels.csv'
    if not labels_csv.exists():
        labels_csv = next(iter(fsl_root.rglob('labels.csv')), None)
    if labels_csv is None:
        return {}
    df = pd.read_csv(labels_csv)
    cols = {c.lower(): c for c in df.columns}
    id_col = cols.get('id') or df.columns[0]
    label_col = cols.get('label') or df.columns[1]
    mapping = {}
    for _, row in df.iterrows():
        try:
            clip_id = int(row[id_col])
        except (ValueError, TypeError):
            continue
        action = normalize_label(str(row[label_col]))
        if action:
            mapping[clip_id] = action
    return mapping


def load_from_fsl105(fsl_root: Path):
    sequences, labels = [], []
    exts = {'.mp4', '.avi', '.mov', '.mkv', '.MOV', '.MP4', '.AVI', '.MKV'}
    clips_root = find_clips_root(fsl_root)
    if clips_root is None:
        raise FileNotFoundError('Could not find fsl_clips/ — re-run download cell')
    id_to_action = load_id_to_action(fsl_root)
    sign_counts = {a: 0 for a in ACTIONS}

    for clip_id, action in id_to_action.items():
        if action not in ACTIONS:
            continue
        folder = clips_root / str(clip_id)
        if not folder.is_dir():
            continue
        label_idx = int(np.where(ACTIONS == action)[0][0])
        for vid in folder.iterdir():
            if vid.suffix in exts:
                seq = video_to_sequence(vid)
                if seq is not None:
                    sequences.append(seq)
                    labels.append(label_idx)
                    sign_counts[action] += 1

    print('Clips per sign:', {k: sign_counts[k] for k in ACTIONS})
    return sequences, labels


def find_data_roots(base: Path):
    if base.exists() and (next(base.rglob('labels.csv'), None) or find_clips_root(base)):
        return 'fsl105', base
    candidates = list(base.rglob('MP_Data'))
    if candidates:
        return 'mp_data', candidates[0]
    return None, None


def load_from_mp_data(mp_root: Path):
    sequences, labels = [], []
    for folder in mp_root.iterdir():
        if not folder.is_dir():
            continue
        label = normalize_label(folder.name)
        if label is None:
            continue
        label_idx = int(np.where(ACTIONS == label)[0][0])
        for seq_dir in folder.iterdir():
            if not seq_dir.is_dir():
                continue
            window = [np.load(seq_dir / f'{f}.npy') for f in range(SEQUENCE_LENGTH)
                      if (seq_dir / f'{f}.npy').exists()]
            if len(window) == SEQUENCE_LENGTH:
                sequences.append(window)
                labels.append(label_idx)
    return sequences, labels


if RAW_CHECKPOINT.exists():
    print(f'Loading raw landmark checkpoint: {RAW_CHECKPOINT}')
    raw = np.load(RAW_CHECKPOINT, allow_pickle=True)
    sequences = list(raw['sequences'])
    labels = list(raw['labels'])
    mode = str(raw.get('mode', 'checkpoint'))
else:
    mode, root = find_data_roots(DATA_ROOT)
    if mode == 'fsl105':
        sequences, labels = load_from_fsl105(root)
    elif mode == 'mp_data':
        sequences, labels = load_from_mp_data(root)
    else:
        sequences, labels = []

    if not sequences:
        raise RuntimeError('No training data found — re-run download cell')

    np.savez_compressed(
        RAW_CHECKPOINT,
        sequences=np.array(sequences, dtype=object),
        labels=np.array(labels),
        actions=ACTIONS,
        mode=mode,
    )
    print(f'Saved raw landmark checkpoint -> {RAW_CHECKPOINT}')

print(f'Mode: {mode}, loaded {len(sequences)} sequences')

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical
import joblib

SCALER_CHECKPOINT = CHECKPOINT_DIR / 'scaler.pkl'


def augment_sequence(seq, rng):
    out = seq.copy().astype(np.float32)
    noise = rng.normal(0, 0.015, size=out.shape)
    noise[:, 2::3] *= 0.3
    out += noise
    if rng.random() < 0.5:
        src_len = rng.integers(22, 38)
        idx = np.linspace(0, len(out)-1, src_len)
        stretched = np.array([np.interp(idx, np.arange(len(out)), out[:, j]) for j in range(out.shape[1])]).T
        new_idx = np.linspace(0, len(stretched)-1, SEQUENCE_LENGTH)
        out = np.array([np.interp(new_idx, np.arange(len(stretched)), stretched[:, j]) for j in range(stretched.shape[1])]).T
    pose_end, lh_end = 132, 132 + 63
    if rng.random() < 0.25:
        out[:, pose_end:lh_end] = 0
    if rng.random() < 0.25:
        out[:, lh_end:] = 0
    scale = rng.uniform(0.92, 1.08)
    out[:, 0::3] *= scale
    out[:, 1::3] *= scale
    return out


if PREPROCESSED_CHECKPOINT.exists() and SCALER_CHECKPOINT.exists():
    print(f'Loading preprocessed checkpoint: {PREPROCESSED_CHECKPOINT}')
    prep = np.load(PREPROCESSED_CHECKPOINT)
    X_train = prep['X_train']
    X_test = prep['X_test']
    y_train = prep['y_train']
    y_test = prep['y_test']
    scaler = joblib.load(SCALER_CHECKPOINT)
    y_train_cat = to_categorical(y_train, NUM_CLASSES)
    y_test_cat = to_categorical(y_test, NUM_CLASSES)
    print('Train:', X_train.shape, 'Test:', X_test.shape, '(restored from checkpoint)')
else:
    rng = np.random.default_rng(42)
    X_base = np.array(sequences, dtype=np.float32)
    y = np.array(labels, dtype=int)
    X_list, y_list = [], []
    for seq, label in zip(X_base, y):
        X_list.append(seq)
        y_list.append(label)
        for _ in range(3):
            X_list.append(augment_sequence(seq, rng))
            y_list.append(label)
    X = np.array(X_list)
    y = np.array(y_list)
    print('Augmented shape:', X.shape)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.15, random_state=42, stratify=y,
    )
    scaler = StandardScaler()
    scaler.fit(X_train.reshape(-1, KEYPOINT_DIM))
    X_train = scaler.transform(X_train.reshape(-1, KEYPOINT_DIM)).reshape(X_train.shape)
    X_test = scaler.transform(X_test.reshape(-1, KEYPOINT_DIM)).reshape(X_test.shape)
    y_train_cat = to_categorical(y_train, NUM_CLASSES)
    y_test_cat = to_categorical(y_test, NUM_CLASSES)

    np.savez_compressed(
        PREPROCESSED_CHECKPOINT,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        actions=ACTIONS,
    )
    joblib.dump(scaler, SCALER_CHECKPOINT)
    print(f'Saved preprocessed checkpoint -> {PREPROCESSED_CHECKPOINT}')
    print(f'Saved scaler -> {SCALER_CHECKPOINT}')
    print('Train:', X_train.shape, 'Test:', X_test.shape)

In [ ]:
# Preprocessed data loaded/saved in the cell above.
# Delete 02_preprocessed_augmented.npz to force re-augmentation.
print('Ready for training:', X_train.shape, X_test.shape)

## Transformer encoder (VSL — Video Sign Language landmarks)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

D_MODEL = 128
NUM_HEADS = 4
FF_DIM = 256
NUM_LAYERS = 3
DROPOUT = 0.3


class PositionalEmbedding(layers.Layer):
    def __init__(self, seq_len, d_model, **kwargs):
        super().__init__(**kwargs)
        self.seq_len = seq_len
        self.d_model = d_model
        self.project = layers.Dense(d_model)
        self.pos_emb = layers.Embedding(input_dim=seq_len, output_dim=d_model)

    def call(self, x):
        x = self.project(x)
        positions = tf.range(start=0, limit=self.seq_len, delta=1)
        return x + self.pos_emb(positions)


def transformer_encoder(x, head_size, num_heads, ff_dim, dropout=0.3):
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=head_size, dropout=dropout)(x, x)
    attn = layers.Dropout(dropout)(attn)
    x1 = layers.LayerNormalization(epsilon=1e-6)(x + attn)
    ff = layers.Dense(ff_dim, activation='relu')(x1)
    ff = layers.Dense(x1.shape[-1])(ff)
    ff = layers.Dropout(dropout)(ff)
    return layers.LayerNormalization(epsilon=1e-6)(x1 + ff)


def build_transformer_model():
    inputs = keras.Input(shape=(SEQUENCE_LENGTH, KEYPOINT_DIM))
    x = PositionalEmbedding(SEQUENCE_LENGTH, D_MODEL)(inputs)
    x = layers.Dropout(DROPOUT)(x)
    for _ in range(NUM_LAYERS):
        x = transformer_encoder(x, D_MODEL // NUM_HEADS, NUM_HEADS, FF_DIM, DROPOUT)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation='relu', kernel_regularizer=keras.regularizers.l2(0.001))(x)
    x = layers.Dropout(DROPOUT)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    m = keras.Model(inputs, outputs, name='fsl_vsl_transformer')
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
    return m


BEST_MODEL_PATH = TRAINING_DIR / 'best_model.keras'
if BEST_MODEL_PATH.exists():
    print(f'Found training checkpoint — will resume in next cell: {BEST_MODEL_PATH}')
else:
    model = build_transformer_model()
    model.summary()

In [ ]:
import json
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger

LATEST_MODEL_PATH = TRAINING_DIR / 'latest_model.keras'
HISTORY_PATH = TRAINING_DIR / 'training_history.json'

RESUME_TRAINING = BEST_MODEL_PATH.exists()

if RESUME_TRAINING:
    print(f'Resuming from checkpoint: {BEST_MODEL_PATH}')
    model = keras.models.load_model(
        str(BEST_MODEL_PATH),
        custom_objects={'PositionalEmbedding': PositionalEmbedding},
    )
else:
    print('Training new transformer model from scratch')
    model = build_transformer_model()

callbacks = [
    ModelCheckpoint(str(BEST_MODEL_PATH), monitor='val_accuracy', save_best_only=True, verbose=1),
    ModelCheckpoint(str(LATEST_MODEL_PATH), save_best_only=False, verbose=0),
    CSVLogger(str(TRAINING_DIR / 'training_log.csv'), append=RESUME_TRAINING),
    EarlyStopping(patience=20, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.5, patience=8, min_lr=1e-6),
]

history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_test, y_test_cat),
    epochs=150,
    batch_size=32,
    callbacks=callbacks,
)

model.load_weights(str(BEST_MODEL_PATH))
model.save(TRAINING_DIR / 'final_model.keras')
with open(HISTORY_PATH, 'w') as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history.history.items()}, f, indent=2)

print(f'Saved best model  -> {BEST_MODEL_PATH}')
print(f'Saved final model -> {TRAINING_DIR / "final_model.keras"}')
print(f'Saved history     -> {HISTORY_PATH}')

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
print('Test accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=ACTIONS))

In [ ]:
export_dir = Path('/content/fsl_15_transformer_export')
export_dir.mkdir(exist_ok=True)

model.save(export_dir / 'fsl_15_transformer_model.keras')
np.save(export_dir / 'action_labels_15.npy', ACTIONS)
joblib.dump(scaler, export_dir / 'scaler.pkl')

!cd {CHECKPOINT_DIR.parent} && zip -r /content/fsl_15_transformer_checkpoints.zip {CHECKPOINT_DIR.name}

print('Exported model  ->', export_dir)
print('All checkpoints ->', CHECKPOINT_DIR)
print('Checkpoint zip  -> /content/fsl_15_transformer_checkpoints.zip')
!ls -la /content/fsl_15_transformer_export
!ls -la {CHECKPOINT_DIR}
!ls -la {TRAINING_DIR}

In [ ]:
from google.colab import files

for fname in ['fsl_15_transformer_model.keras', 'action_labels_15.npy', 'scaler.pkl']:
    files.download(str(export_dir / fname))

files.download('/content/fsl_15_transformer_checkpoints.zip')